In [1]:
pip install qiskit qiskit-aer qiskit-ibm-runtime qiskit-ibm-provider


  Using cached qiskit-2.2.3-cp39-abi3-macosx_11_0_arm64.whl.metadata (12 kB)
  Using cached qiskit_aer-0.17.2-cp313-cp313-macosx_11_0_arm64.whl.metadata (8.3 kB)
  Using cached qiskit_ibm_runtime-0.44.0-py3-none-any.whl.metadata (21 kB)
  Using cached qiskit_ibm_provider-0.11.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached rustworkx-0.17.1-cp39-abi3-macosx_11_0_arm64.whl.metadata (10 kB)
  Using cached numpy-2.4.0-cp313-cp313-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached scipy-1.16.3-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached dill-0.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached stevedore-5.6.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached requests_ntlm-1.3.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached urllib3-2.6.2-py3-none-any.whl.metadata (6.6 kB)
  Using cached ibm_platform_services-0.72.0-py

In [1]:
# qstream_qiskit_demo.py

import math
import hashlib
from dataclasses import dataclass
from typing import List, Tuple

from qiskit import QuantumCircuit

# ---------- 1. Quantum RNG using Qiskit ----------

def qrng_bits(num_bytes: int) -> bytes:
    """
    Generate num_bytes of randomness using single-qubit Hadamard measurements.
    Each circuit shot gives 1 random bit; we batch shots for efficiency.
    """
    num_bits = num_bytes * 8
    # build 1-qubit H+measure circuit
    qc = QuantumCircuit(1)
    qc.h(0)
    qc.measure_all()

    # Sampler gives probabilities, not per-shot results, so instead we use Aer directly:
    from qiskit_aer import Aer
    from qiskit import transpile

    backend = Aer.get_backend("qasm_simulator")
    tqc = transpile(qc, backend)
    counts = backend.run(tqc, shots=num_bits).result().get_counts()

    # expand counts dict into bitstring
    bits = []
    for bitval, count in counts.items():
        bits.extend(bitval * count)
    # bits list like ['0', '1', '0', ...]; truncate to needed bits
    bits = bits[:num_bits]
    # pack into bytes
    out = bytearray()
    for i in range(0, num_bits, 8):
        chunk = bits[i:i+8]
        val = 0
        for b in chunk:
            val = (val << 1) | (1 if b == '1' else 0)
        out.append(val)
    return bytes(out)


# ---------- 2. Helper primitives ----------

def h(data: bytes) -> bytes:
    return hashlib.sha256(data).digest()

def xor_bytes(a: bytes, b: bytes) -> bytes:
    return bytes(x ^ y for x, y in zip(a, b))


# ---------- 3. rUID and registration ----------

@dataclass
class UserState:
    name: str
    otc: bytes
    U: bytes
    counter: int = 0
    last_LM: bytes = b""

    def next_ruid(self) -> bytes:
        self.counter += 1
        data = self.otc + self.U + self.counter.to_bytes(8, "big") + self.last_LM
        return h(data)

def register_user(name: str, email: str) -> UserState:
    otc = qrng_bits(32)        # 256-bit OTC from quantum RNG
    U = email.encode()
    return UserState(name=name, otc=otc, U=U)


# ---------- 4. q-stream server simulation ----------

@dataclass
class QStreamServer:
    block_bits: int = 2_097_152
    num_blocks: int = 18
    min_kgen_bits: int = 128
    max_kgen_bits: int = 256

    def generate_R_and_metadata(
        self,
        users: List[UserState]
    ) -> Tuple[bytes, dict]:
        block_bytes = self.block_bits // 8
        R = bytearray(qrng_bits(block_bytes))   # QRNG block from Qiskit
        meta = {u.name: [] for u in users}

        cursor = 0
        for ordinal in range(self.num_blocks):
            length_bits = self.min_kgen_bits + (
                qrng_bits(2)[0] % (self.max_kgen_bits - self.min_kgen_bits + 1)
            )
            length_bytes = math.ceil(length_bits / 8)
            if cursor + length_bytes >= block_bytes:
                cursor = 0

            start = cursor
            cursor += length_bytes

            for u in users:
                ruid = u.next_ruid()
                P = (
                    start.to_bytes(4, "big")
                    + length_bytes.to_bytes(2, "big")
                    + ordinal.to_bytes(1, "big")
                )
                S = ruid[: len(P)]
                PS = xor_bytes(P, S)

                # small random offset derived from quantum bytes
                offset = qrng_bits(1)[0] % 32
                pos = min(start + offset, block_bytes - len(ruid) - len(PS))

                R[pos:pos + len(ruid)] = ruid
                R[pos + len(ruid):pos + len(ruid) + len(PS)] = PS

                meta[u.name].append((start, length_bytes, ordinal))

        return bytes(R), meta


# ---------- 5. Client key extraction + key generation ----------

def extract_kgen_for_user(
    R: bytes,
    user: UserState,
    num_blocks: int = 18,
) -> List[bytes]:
    kgen_blocks = []
    search_from = 0
    for _ in range(num_blocks):
        ruid = user.next_ruid()
        idx = R.find(ruid, search_from)
        if idx == -1:
            break
        ps_start = idx + len(ruid)
        ps_end = ps_start + 7
        PS = R[ps_start:ps_end]
        S = ruid[: len(PS)]
        P = xor_bytes(PS, S)

        start = int.from_bytes(P[0:4], "big")
        length = int.from_bytes(P[4:6], "big")
        ordinal = P[6]

        kgen = R[start:start + length]
        kgen_blocks.append((ordinal, kgen))

        user.last_LM = kgen
        search_from = ps_end

    kgen_blocks.sort(key=lambda x: x[0])
    return [blk for _, blk in kgen_blocks]


def generate_key_from_kgen(kgen_blocks: List[bytes], key_bytes: int) -> bytes:
    master = b""
    blocks = kgen_blocks[:]
    while len(master) < key_bytes:
        concat = b"".join(blocks) + master
        hval = h(concat)
        master += hval
        blocks = blocks[1:] + blocks[:1]
    return master[:key_bytes]


# ---------- 6. Simple OTP encrypt/decrypt demo ----------

def otp(key: bytes, msg: bytes) -> bytes:
    return xor_bytes(key, msg)

def demo():
    alice = register_user("Alice", "alice@navy.example")
    bob   = register_user("Bob",   "bob@navy.example")

    server = QStreamServer()
    R, meta = server.generate_R_and_metadata([alice, bob])

    alice_kgen = extract_kgen_for_user(R, alice)
    bob_kgen   = extract_kgen_for_user(R, bob)

    key_len = 64
    KA = generate_key_from_kgen(alice_kgen, key_len)
    KB = generate_key_from_kgen(bob_kgen,   key_len)

    assert KA == KB, "Key mismatch"

    msg_str = input("Quantum-safe naval message.") # Get string input
    msg = msg_str.encode('utf-8') # Encode the string to bytes
    key_for_msg = KA[:len(msg)]
    ct = otp(key_for_msg, msg)
    pt_bytes = otp(KB[:len(msg)], ct) # Decrypt returns bytes
    pt = pt_bytes.decode('utf-8') # Decode bytes back to string for printing

    print("Ciphertext (hex):", ct.hex())
    print("Recovered:        ", pt)

if __name__ == "__main__":
    demo()


Ciphertext (hex): 88db
Recovered:         kk
